# ⚪ Silver - Classificação Brasileirão 2026

## Descrição
Este notebook limpa e padroniza os dados da camada Bronze.

### Pipeline ETL
- **Extract**: Lê do Parquet Bronze
- **Transform**: Limpeza, padronização, tipos de dados
- **Load**: Salva em `Files/silver/classificacao.parquet`

In [ ]:
# Import libraries
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Extract - Read from Bronze
logger.info("Lendo dados da camada Bronze...")
df = pd.read_parquet("Files/bronze/classificacao.parquet")
logger.info(f"Dados lidos: {len(df)} registros")
df.head()

In [ ]:
# Transform - Limpeza e padronização
logger.info("Iniciando transformações...")

df_silver = df.copy()

# 1. Padronizar nomes das colunas (snake_case)
df_silver.columns = [col.lower().replace(" ", "_") for col in df_silver.columns]
logger.info(f"Colunas padronizadas: {list(df_silver.columns)}")

# 2. Converter tipos de dados
numeric_cols = ["j", "v", "e", "d", "gp", "gc", "sg", "pts"]
for col in numeric_cols:
    df_silver[col] = df_silver[col].astype("int32")

df_silver["posição"] = df_silver["posição"].astype("int8")

# 3. Limpar nomes de times (trim, uppercase)
df_silver["time"] = df_silver["time"].str.strip().str.title()

# 4. Adicionar metadata
from datetime import datetime
df_silver["extracted_at"] = datetime.now().isoformat()
df_silver["camada"] = "silver"

logger.info(f"Transformações aplicadas: {len(df_silver)} registros")
df_silver.dtypes

In [ ]:
# Load - Save to Silver Parquet
output_path = "Files/silver/classificacao.parquet"
import os
os.makedirs("Files/silver", exist_ok=True)
df_silver.to_parquet(output_path, index=False)
logger.info(f"Dados salvos em: {output_path}")

# Display final result
df_silver